In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_5.py ──
"""
Shared infrastructure for MLFP03 Exercise 5 — Class Imbalance & Calibration.

Contains: Singapore credit scoring data loader, Kailash PreprocessingPipeline
wiring, cost-matrix constants, metric helpers, reliability-diagram helpers,
and an OUTPUT_DIR convention for every technique file to write visual proof
to the same place.

Technique-specific code (SMOTE call, focal loss gradient, Platt/Isotonic
model wiring) does NOT belong here — it lives in the per-technique files.
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY — every technique writes visual proof to the same place
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "ex5_imbalance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# BUSINESS CONTEXT — Singapore retail bank credit scoring
# ════════════════════════════════════════════════════════════════════════
# These constants drive every technique file. A 100:1 cost ratio is
# realistic for SEA consumer lending: the average charged-off unsecured
# loan in Singapore is ~S$10,000 (MAS consumer credit report 2024), and
# the operational cost of a false decline (manual review + lost NPV of
# the customer relationship) is roughly S$100.


@dataclass(frozen=True)
class CostMatrix:
    """Dollar cost of each confusion-matrix cell.

    fn = cost of missing a default (charge-off loss)
    fp = cost of a false alarm (manual review + lost relationship NPV)
    """

    fn: float = 10_000.0
    fp: float = 100.0

    @property
    def optimal_threshold(self) -> float:
        """Bayes-optimal threshold for this cost matrix: t* = fp / (fp + fn)."""
        return self.fp / (self.fp + self.fn)

    def total_cost(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        return float(fp * self.fp + fn * self.fn)


DEFAULT_COSTS = CostMatrix(fn=10_000.0, fp=100.0)

# Annual volume for ROI analysis — calibrated to a mid-tier SG retail bank.
ANNUAL_APPLICATIONS = 100_000


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring dataset
# ════════════════════════════════════════════════════════════════════════
# The dataset is loaded through the MLFPDataLoader so it works identically
# in local (.data_cache) and Colab (Drive + gdown) formats.


def load_credit_splits(
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    """Load the SG credit scoring dataset and return (X_train, y_train, X_test, y_test, pos_rate).

    Uses kailash-ml PreprocessingPipeline for consistent preprocessing across
    every technique file. Returns numpy arrays ready for sklearn-style fit.
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet")

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        credit,
        target="default",
        seed=seed,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_cols = [c for c in result.train_data.columns if c != "default"]
    X_train, y_train, _ = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    pos_rate = float(y_train.mean())
    return X_train, y_train, X_test, y_test, pos_rate


# ════════════════════════════════════════════════════════════════════════
# METRIC HELPERS — complete taxonomy in one call
# ════════════════════════════════════════════════════════════════════════


def metrics_row(
    name: str,
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, Any]:
    """Compute a full metrics row for a given strategy name + probabilities.

    Returns a dict compatible with polars DataFrame construction so every
    technique file can build comparison tables with identical column shapes.
    """
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        "strategy": name,
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def print_metrics_table(rows: list[dict[str, Any]], title: str) -> None:
    """Print a comparison table across strategies."""
    print(f"\n{'=' * 70}")
    print(f"  {title}")
    print(f"{'=' * 70}")
    print(
        f"{'Strategy':<22} {'AUC-PR':>8} {'Brier':>8} {'F1':>8} "
        f"{'Precision':>10} {'Recall':>8}"
    )
    print("─" * 70)
    for r in rows:
        print(
            f"{r['strategy']:<22} {r['auc_pr']:>8.4f} {r['brier']:>8.4f} "
            f"{r['f1']:>8.4f} {r['precision']:>10.4f} {r['recall']:>8.4f}"
        )


def rows_to_dataframe(rows: list[dict[str, Any]]) -> pl.DataFrame:
    """Convert metrics rows to a polars DataFrame (for persistence / plots)."""
    return pl.DataFrame(rows)


# ════════════════════════════════════════════════════════════════════════
# RELIABILITY DIAGRAM DATA (calibration curve, binned)
# ════════════════════════════════════════════════════════════════════════


def reliability_bins(
    y_true: np.ndarray, y_proba: np.ndarray, n_bins: int = 10
) -> pl.DataFrame:
    """Compute a binned reliability diagram as a polars DataFrame.

    Columns: bin_lower, bin_upper, mean_pred, empirical_rate, count, gap
    """
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (y_proba >= lo) & (y_proba < hi if i < n_bins - 1 else y_proba <= hi)
        count = int(mask.sum())
        if count == 0:
            continue
        mean_pred = float(y_proba[mask].mean())
        empirical = float(y_true[mask].mean())
        rows.append(
            {
                "bin_lower": float(lo),
                "bin_upper": float(hi),
                "mean_pred": mean_pred,
                "empirical_rate": empirical,
                "count": count,
                "gap": float(abs(mean_pred - empirical)),
            }
        )
    return pl.DataFrame(rows)


def print_reliability(name: str, bins: pl.DataFrame) -> None:
    print(f"\n  Reliability bins — {name}")
    print(f"  {'mean_pred':>10} {'empirical':>10} {'|gap|':>8} {'n':>6}")
    print("  " + "─" * 38)
    for row in bins.iter_rows(named=True):
        print(
            f"  {row['mean_pred']:>10.3f} {row['empirical_rate']:>10.3f} "
            f"{row['gap']:>8.3f} {row['count']:>6}"
        )


# ════════════════════════════════════════════════════════════════════════
# BUSINESS ROI CALCULATOR
# ════════════════════════════════════════════════════════════════════════


def annual_roi(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float,
    costs: CostMatrix = DEFAULT_COSTS,
    annual_volume: int = ANNUAL_APPLICATIONS,
) -> dict[str, float]:
    """Project test-set confusion matrix onto an annual volume.

    Returns a dict with caught_defaults, missed_defaults, false_alarms,
    model_cost, no_model_cost, annual_savings — all in dollars.
    """
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    pos_rate = float(y_true.mean())
    scale = annual_volume / len(y_true)
    annual_fn = fn * scale
    annual_fp = fp * scale
    annual_tp = tp * scale
    n_defaults_annual = pos_rate * annual_volume
    model_cost = annual_fn * costs.fn + annual_fp * costs.fp
    no_model_cost = n_defaults_annual * costs.fn
    return {
        "threshold": float(threshold),
        "defaults_caught": float(annual_tp),
        "defaults_missed": float(annual_fn),
        "false_alarms": float(annual_fp),
        "model_cost_usd": float(model_cost),
        "no_model_cost_usd": float(no_model_cost),
        "annual_savings_usd": float(no_model_cost - model_cost),
        "annual_volume": int(annual_volume),
    }


def print_roi(name: str, roi: dict[str, float]) -> None:
    print(f"\n  ROI — {name}")
    print(f"    Threshold:          {roi['threshold']:.4f}")
    print(f"    Defaults caught:    {roi['defaults_caught']:>12,.0f}")
    print(f"    Defaults missed:    {roi['defaults_missed']:>12,.0f}")
    print(f"    False alarms:       {roi['false_alarms']:>12,.0f}")
    print(f"    Model cost:         ${roi['model_cost_usd']:>12,.0f}")
    print(f"    No-model cost:      ${roi['no_model_cost_usd']:>12,.0f}")
    print(f"    Annual savings:     ${roi['annual_savings_usd']:>12,.0f}")


# ════════════════════════════════════════════════════════════════════════
# PERSISTENCE — shared parquet of per-strategy probabilities
# ════════════════════════════════════════════════════════════════════════
# Each technique file writes its y_proba vector to a parquet under
# OUTPUT_DIR so that later technique files (threshold opt, calibration,
# final comparison) can read them without re-training.

PROBA_STORE = OUTPUT_DIR / "strategy_probabilities.parquet"


def save_strategy_proba(name: str, y_proba: np.ndarray) -> None:
    """Upsert a strategy's probability vector into the shared parquet store."""
    new_df = pl.DataFrame({"strategy": [name] * len(y_proba), "y_proba": y_proba})
    if PROBA_STORE.exists():
        existing = pl.read_parquet(PROBA_STORE)
        existing = existing.filter(pl.col("strategy") != name)
        combined = pl.concat([existing, new_df])
    else:
        combined = new_df
    combined.write_parquet(PROBA_STORE)


def load_strategy_proba(name: str) -> np.ndarray:
    """Read back a previously-saved probability vector for a strategy."""
    if not PROBA_STORE.exists():
        raise FileNotFoundError(
            f"{PROBA_STORE} not found — run the earlier technique files first."
        )
    df = pl.read_parquet(PROBA_STORE).filter(pl.col("strategy") == name)
    if df.height == 0:
        raise KeyError(f"Strategy {name!r} not found in {PROBA_STORE}")
    return df["y_proba"].to_numpy()


def list_saved_strategies() -> list[str]:
    if not PROBA_STORE.exists():
        return []
    df = pl.read_parquet(PROBA_STORE)
    return df["strategy"].unique().to_list()




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 5.3: Loss Functions — Focal Loss & Alpha Weighting
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - The focal loss equation FL(p_t) = -(1-p_t)^gamma * log(p_t)
#   - What the gamma parameter does intuitively (down-weight easy examples)
#   - How alpha-weighting approximates focal loss with any boosted learner
#   - How to sweep alpha and read the sensitivity curve
#
# PREREQUISITES: 02_sampling_strategies.py (cost-sensitive baseline saved)
# ESTIMATED TIME: ~25 min
#
# 5-PHASE STRUCTURE:
#   Theory   — focal loss derivation + gamma intuition
#   Build    — LightGBM with a family of alpha multipliers
#   Train    — sweep alpha in [0.5, 1, 2, 5, 10]
#   Visualise — AUC-PR and Brier vs alpha (sensitivity curve)
#   Apply    — OCBC SME-loan default detection (long-tail hard cases)
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import lightgbm as lgb
import numpy as np
import plotly.graph_objects as go
import polars as pl
from dotenv import load_dotenv


load_dotenv()



## THEORY — Focal Loss and the Gamma Knob


In [ ]:
# Cross-entropy treats every example with the same weight. A well-classified
# example (the model says p=0.99 and the label is 1) contributes almost as
# much to the average loss as a hard example (p=0.51 where the model is
# unsure). That wastes capacity on easy wins.
#
# Focal Loss (Lin et al., 2017, ICCV) adds a modulating factor (1-p_t)^gamma:
#
#     FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
#
# where p_t = p if y=1 else 1-p. When the model is already confident and
# correct, (1 - p_t) is tiny and the example essentially vanishes from
# the loss. Hard examples (p_t around 0.5) still contribute their full
# weight. Gamma=0 recovers cross-entropy; gamma=2 is the canonical setting.
#
# WHY THIS HELPS IMBALANCED DATA: the majority class usually has many
# easy examples. Focal loss automatically down-weights them without
# any resampling. Combined with alpha (the class prior), you get a
# loss function that focuses gradient on the examples that actually
# move the decision boundary.
#
# LIGHTGBM APPROXIMATION: LightGBM's default objective doesn't expose
# focal loss directly. We approximate it by sweeping an alpha multiplier
# on top of `scale_pos_weight`. This captures the "focus on hard examples"
# effect in a single tunable parameter.



## BUILD + TRAIN — alpha sweep


In [ ]:
X_train, y_train, X_test, y_test, pos_rate = load_credit_splits()
base_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

print("\n" + "=" * 70)
print("  Exercise 5.3 — Focal Loss (Alpha Sweep Approximation)")
print("=" * 70)
print(f"  Base scale_pos_weight (class-balanced): {base_weight:.2f}")

alpha_multipliers = [0.5, 1.0, 2.0, 5.0, 10.0]
metric_rows: list[dict] = []

for alpha in alpha_multipliers:
    pos_w = alpha * base_weight
    model = lgb.LGBMClassifier(
        n_estimators=300,
        scale_pos_weight=pos_w,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    name = f"focal_alpha_{alpha:.1f}"
    save_strategy_proba(name, y_proba)
    row = metrics_row(f"alpha={alpha:.1f}", y_test, y_proba)
    row["alpha"] = float(alpha)
    metric_rows.append(row)



### Checkpoint 3


In [ ]:
assert len(metric_rows) == len(alpha_multipliers), "Must sweep every alpha"
assert all(0 <= r["auc_pr"] <= 1 for r in metric_rows), "AUC-PR in [0,1]"
assert all(0 <= r["brier"] <= 1 for r in metric_rows), "Brier in [0,1]"
print("[ok] Checkpoint 3 — alpha sweep complete\n")



## VISUALISE — AUC-PR and Brier sensitivity to alpha


In [ ]:
print_metrics_table(metric_rows, "Focal loss alpha sweep")

best_by_pr = max(metric_rows, key=lambda r: r["auc_pr"])
best_by_brier = min(metric_rows, key=lambda r: r["brier"])

print("\n  Sensitivity summary:")
print(
    f"    Best AUC-PR: alpha={best_by_pr['alpha']:.1f} (AUC-PR={best_by_pr['auc_pr']:.4f})"
)
print(
    f"    Best Brier:  alpha={best_by_brier['alpha']:.1f} "
    f"(Brier={best_by_brier['brier']:.4f})"
)

# Persist for later files
pl.DataFrame(metric_rows).write_parquet(OUTPUT_DIR / "focal_sweep_metrics.parquet")
print(f"\n  Saved: {OUTPUT_DIR / 'focal_sweep_metrics.parquet'}")



### Visual: Focal loss alpha sweep — AUC-PR and Brier vs alpha


In [ ]:
alphas = [r["alpha"] for r in metric_rows]
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=alphas,
        y=[r["auc_pr"] for r in metric_rows],
        mode="lines+markers",
        name="AUC-PR",
        marker=dict(size=10),
        line=dict(color="#6366f1", width=3),
    )
)
fig.add_trace(
    go.Scatter(
        x=alphas,
        y=[r["brier"] for r in metric_rows],
        mode="lines+markers",
        name="Brier score",
        marker=dict(size=10),
        line=dict(color="#f43f5e", width=3),
        yaxis="y2",
    )
)
fig.update_layout(
    title="Focal Loss Alpha Sweep: ranking (AUC-PR) vs calibration (Brier)",
    xaxis_title="Alpha multiplier",
    yaxis=dict(title="AUC-PR (higher = better ranking)"),
    yaxis2=dict(
        title="Brier (lower = better calibration)",
        overlaying="y",
        side="right",
    ),
    height=450,
    legend=dict(orientation="h", y=-0.2),
)
viz_path = OUTPUT_DIR / "ex5_03_focal_alpha_sweep.html"
fig.write_html(str(viz_path))
print(f"  Saved: {viz_path}")



### Visual: Focal loss curve for different gamma values


In [ ]:
p_t = np.linspace(0.01, 0.99, 200)
fig2 = go.Figure()
for gamma in [0, 0.5, 1, 2, 5]:
    fl = -((1 - p_t) ** gamma) * np.log(p_t)
    fig2.add_trace(
        go.Scatter(x=p_t, y=fl, mode="lines", name=f"gamma={gamma}", line=dict(width=2))
    )
fig2.update_layout(
    title="Focal Loss Curve: FL(p_t) = -(1-p_t)^gamma * log(p_t)",
    xaxis_title="p_t (model confidence for correct class)",
    yaxis_title="Focal loss",
    height=450,
    legend=dict(orientation="h", y=-0.2),
)
viz_path2 = OUTPUT_DIR / "ex5_03_focal_loss_curves.html"
fig2.write_html(str(viz_path2))
print(f"  Saved: {viz_path2}")

# INTERPRETATION: Usually AUC-PR peaks at a DIFFERENT alpha from Brier.
# AUC-PR rewards ranking quality; Brier rewards calibration. The two
# objectives pull in opposite directions — higher alpha pushes the
# model to over-predict the positive class, improving recall but
# worsening calibration. This is the core trade-off in 5.5 (where we
# will calibrate explicitly rather than trying to squeeze it out of
# the loss function alone).



## APPLY — OCBC SME-loan default detection


In [ ]:
# SCENARIO: OCBC Singapore's SME portfolio has ~18,000 active business
# loans to local SMEs (F&B, retail, manufacturing). The underwriting
# team wants an early-warning model that flags borrowers likely to
# default in the next 90 days.
#
# The imbalance is severe AND heterogeneous:
#   - Overall default rate: ~3% (332 defaults / year)
#   - Long-tail: most defaults come from obvious distress signals
#     (missed payroll, rent arrears, bounced cheques) which are easy
#     to classify. The HARD cases are SMEs that look healthy until
#     the final month — these are the ones a good early-warning
#     system must catch.
#
# Why focal loss matters here:
#   - Standard cost-sensitive training spreads gradient evenly across
#     all defaulters, including the "obvious" ones the model already
#     nails at p=0.99. Gradient on those is wasted.
#   - Focal loss (high gamma/alpha) automatically shifts gradient onto
#     the borderline cases. The model learns subtler distress signals
#     (working capital compression, supplier concentration drift)
#     because that's where the loss lives.
#
# BUSINESS IMPACT: early warning 90 days before default is worth
# S$30,000 per loan in avoided write-off (MAS SME credit report 2024:
# recovery rate jumps from 18% to 47% with 90-day notice because the
# bank can restructure or unwind facilities before the collateral
# value deteriorates). Capturing even 40 extra borderline cases per
# year = S$1.2M/year recovered. The alpha sweep is what finds that
# capture rate.

print("\n  OCBC SME early-warning implication:")
print(f"    Alpha sweep tested {len(alpha_multipliers)} points")
print(f"    Best recall at alpha={best_by_pr['alpha']:.1f}: {best_by_pr['recall']:.2%}")
print("    Each additional recalled borderline borrower -> ~S$30K recovered")



## REFLECTION


[x] Derived focal loss FL(p_t) = -(1-p_t)^gamma * log(p_t)
  [x] Understood the gamma/alpha intuition: down-weight easy examples
  [x] Swept alpha in [0.5, 1, 2, 5, 10] and recorded the sensitivity curve
  [x] Observed that AUC-PR and Brier usually peak at DIFFERENT alphas
  [x] Tied the pattern to OCBC SME borderline-default detection

  KEY INSIGHT: The loss function is a knob. Sweeping alpha reveals the
  ranking-vs-calibration trade-off that no single metric can express.

  Next: 04_threshold_optimisation.py — instead of tuning the loss, tune
  the DECISION threshold from the business cost matrix directly.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — 5.3")
print("=" * 70)
print(
)

